In [ ]:
# %%
# ==========================================================
# 1. IMPORTS ET CONFIGURATION
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Style professionnel
sns.set_theme(style="whitegrid")

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'


# %%
# ==========================================================
# 2. CHARGEMENT DES DONNÉES
# ==========================================================

path_processed = os.path.join('..', 'data', 'processed')

file_path = os.path.join(path_processed, 'ci_education_15_35.csv')

# Chargement
_df = pd.read_csv(file_path)

# Vérification rapide
print("Aperçu des données :")
print(_df.head())


# %%
# ==========================================================
# 3. CONSTRUCTION DE L’INDICATEUR DE VULNÉRABILITÉ
# ==========================================================

# Vulnérabilité = chômage + inactivité
_df['vulnerability'] = _df['share_unemp'] + _df['share_inact']

# Agrégation nationale par année
trend = (
    _df
    .groupby('annee')['vulnerability']
    .mean()
    .reset_index()
    .sort_values('annee')
)

print("\nTendance historique :")
print(trend)



# ==========================================================
# 4. FEATURE ENGINEERING TEMPOREL
# ==========================================================

# Création d’un lag temporel
# vulnérabilité(t-1)
trend['lag_1'] = trend['vulnerability'].shift(1)

# Suppression de la première ligne (NaN)
trend = trend.dropna().reset_index(drop=True)

# Variables d’entrée
X = trend[['annee', 'lag_1']]

y = trend['vulnerability']


# %%
# ==========================================================
# 5. ENTRAÎNEMENT DU MODÈLE
# ==========================================================

model = LinearRegression()
model.fit(X, y)

# Prédictions historiques
hist_pred = model.predict(X)

# Évaluation simple
mae = mean_absolute_error(y, hist_pred)

print(f"\nErreur absolue moyenne (MAE) : {mae:.4f}")


# %%
# ==========================================================
# 6. PROJECTIONS 2025–2035
# ==========================================================

future_years = list(range(2025, 2036))

# Dernière valeur connue
last_value = trend['vulnerability'].iloc[-1]


# ==========================================================
# SCÉNARIO 1 : INERTIE STRUCTURELLE
# ==========================================================

scenario_1 = []

current_value = last_value

for year in future_years:

    X_future = pd.DataFrame({
        'annee': [year],
        'lag_1': [current_value]
    })

    pred = model.predict(X_future)[0]

    scenario_1.append(pred)

    current_value = pred


# SCÉNARIO 2 : RÉFORMES PARTIELLES

scenario_2 = []

current_value = last_value

for year in future_years:

    X_future = pd.DataFrame({
        'annee': [year],
        'lag_1': [current_value]
    })

    base_pred = model.predict(X_future)[0]

    # Impact progressif des réformes
    # réduction supplémentaire de 1.2%
    adjusted_pred = base_pred * 0.988

    scenario_2.append(adjusted_pred)

    current_value = adjusted_pred


# SCÉNARIO 3 : TRANSFORMATION STRUCTURELLE

scenario_3 = []

current_value = last_value

for year in future_years:

    X_future = pd.DataFrame({
        'annee': [year],
        'lag_1': [current_value]
    })

    base_pred = model.predict(X_future)[0]

    # Industrialisation + inclusion féminine
    # réduction forte
    adjusted_pred = base_pred * 0.975

    scenario_3.append(adjusted_pred)

    current_value = adjusted_pred



# 7. INTERVALLES D’INCERTITUDE

# Approximation simple basée sur la variance historique
std_dev = trend['vulnerability'].std()

# Intervalle ±
upper_s1 = np.array(scenario_1) + std_dev * 0.35
lower_s1 = np.array(scenario_1) - std_dev * 0.35

upper_s2 = np.array(scenario_2) + std_dev * 0.25
lower_s2 = np.array(scenario_2) - std_dev * 0.25

upper_s3 = np.array(scenario_3) + std_dev * 0.20
lower_s3 = np.array(scenario_3) - std_dev * 0.20



# 8. VISUALISATION STRATÉGIQUE

fig, ax = plt.subplots()


# HISTORIQUE

sns.lineplot(
    data=trend,
    x='annee',
    y='vulnerability',
    marker='o',
    linewidth=3,
    color='#1f2937',
    label='Historique (2015–2025)',
    ax=ax
)


# SCÉNARIO 1

ax.plot(
    future_years,
    scenario_1,
    linestyle='--',
    linewidth=2.5,
    color='#e74c3c',
    label='S1 — Inertie Structurelle'
)

ax.fill_between(
    future_years,
    lower_s1,
    upper_s1,
    color='#e74c3c',
    alpha=0.10
)


# SCÉNARIO 2

ax.plot(
    future_years,
    scenario_2,
    linestyle='-.',
    linewidth=2.5,
    color='#f39c12',
    label='S2 — Réformes Partielles'
)

ax.fill_between(
    future_years,
    lower_s2,
    upper_s2,
    color='#f39c12',
    alpha=0.10
)


# SCÉNARIO 3

ax.plot(
    future_years,
    scenario_3,
    linestyle='-',
    linewidth=3.5,
    color='#27ae60',
    label='S3 — Transformation Structurelle'
)

ax.fill_between(
    future_years,
    lower_s3,
    upper_s3,
    color='#27ae60',
    alpha=0.10
)


# ----------------------------------------------------------
# LIGNE DE RUPTURE
# ----------------------------------------------------------

ax.axvline(
    x=2025,
    linestyle=':',
    color='gray',
    alpha=0.8
)

ax.text(
    2025.2,
    max(upper_s1) - 0.02,
    'Début des scénarios prospectifs',
    fontsize=10,
    style='italic',
    color='gray'
)


# ----------------------------------------------------------
# HABILLAGE
# ----------------------------------------------------------

ax.set_title(
    "Côte d’Ivoire 2035 — Projection de la Vulnérabilité des Jeunes",
    fontsize=16,
    fontweight='bold',
    pad=20
)

ax.set_xlabel(
    "Années",
    fontsize=12
)

ax.set_ylabel(
    "Taux de vulnérabilité",
    fontsize=12
)

ax.set_xlim(2015, 2035)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y))
)

plt.xticks(range(2015, 2036, 2))

plt.legend(
    frameon=True,
    facecolor='white'
)

plt.tight_layout()


# %%
# ==========================================================
# 9. EXPORT DE LA FIGURE
# ==========================================================

output_dir = os.path.join('..', 'outputs', 'figures')

os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, 'projection_vulnerabilite_2035_v2.png')

plt.savefig(save_path, bbox_inches='tight')

print(f"\nFigure sauvegardée : {save_path}")

plt.show()


# %%
# ==========================================================
# 10. TABLEAU RÉCAPITULATIF
# ==========================================================

summary = pd.DataFrame({

    'Scénario': [
        'Inertie Structurelle',
        'Réformes Partielles',
        'Transformation Structurelle'
    ],

    'Projection 2035': [
        f"{scenario_1[-1]:.2%}",
        f"{scenario_2[-1]:.2%}",
        f"{scenario_3[-1]:.2%}"
    ],

    'Réduction depuis 2025': [
        f"{((last_value - scenario_1[-1]) / last_value):.2%}",
        f"{((last_value - scenario_2[-1]) / last_value):.2%}",
        f"{((last_value - scenario_3[-1]) / last_value):.2%}"
    ]
})

print("\n" + "=" * 70)
print("TABLEAU DE BORD PROSPECTIF — HORIZON 2035")
print("=" * 70)

print(summary.to_markdown(index=False))


# %%
# ==========================================================
# 11. INTERPRÉTATION STRATÉGIQUE
# ==========================================================

print("\nINTERPRÉTATION :")
print("- Le scénario d’inertie montre une persistance structurelle de la vulnérabilité.")
print("- Les réformes partielles améliorent progressivement la situation.")
print("- La transformation structurelle produit la baisse la plus significative.")
print("- Les bandes colorées représentent l’incertitude des projections.")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/ci_education_15_35.csv'